## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

In [35]:
from agents import Agent, trace, Runner,function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display

In [36]:
load_dotenv(override=True)

True

In [37]:
from ddgs import DDGS

@function_tool
def ddgs_search(query: str, max_results: int = 5):
    """Search the web using the DuckDuckGo Search Engine"""
    results = DDGS().text(query, max_results=max_results)
    return "\n".join(
        f"- [{result.get('title','')}]({result.get('href','')}): {result.get('body','')}"
        for result in results
    )

# print(ddgs_search("python programming", max_results=5))

In [38]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[ddgs_search],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
HOW_MANY_SEARCHES = 10

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [40]:
HOW_MANY_QUESTIONS = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of clarifying \
questions. Output {HOW_MANY_QUESTIONS} clarifying questions."

class QuestionItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this question is important to the query.")

    question: str = Field(description="The clarifying question to ask.")


class Questions(BaseModel):
    questions: list[QuestionItem] = Field(description="A list of clarifying questions to ask.")


clarifyer_agent = Agent(
    name="ClarifyingQuestionsAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=Questions,
)

In [43]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)


In [ ]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def get_clarifying_questions(query: str):
    """ Use the clarifyer_agent to generate clarifying questions """
    result = await Runner.run(clarifyer_agent, f"Query: {query}")
    return result.final_output

async def run_deep_research(query: str, clarifying_answers: str = ""):
    """ Run the full deep research pipeline """
    full_query = query
    if clarifying_answers:
        full_query = f"{query}\n\nAdditional context from clarifying questions:\n{clarifying_answers}"
    with trace("Research trace"):
        print("Starting research...")
        search_plan = await plan_searches(full_query)
        search_results = await perform_searches(search_plan)
        report = await write_report(full_query, search_results)
        print("Hooray!")
        return report.markdown_report


In [ ]:
import gradio as gr


async def clarify(query: str):
    """ Generate clarifying questions and update the UI """
    if not query.strip():
        return (
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            "", "", "",
            gr.update(visible=False), gr.update(visible=False),
        )
    questions = await get_clarifying_questions(query)
    qs = [qi.question for qi in questions.questions]
    while len(qs) < 3:
        qs.append("")
    return (
        gr.update(label=qs[0], placeholder="Your answer...", value="", visible=bool(qs[0])),
        gr.update(label=qs[1], placeholder="Your answer...", value="", visible=bool(qs[1])),
        gr.update(label=qs[2], placeholder="Your answer...", value="", visible=bool(qs[2])),
        qs[0], qs[1], qs[2],
        gr.update(visible=True),
        gr.update(visible=True),
    )


async def run_research(query: str, ans1: str, ans2: str, ans3: str, q1: str, q2: str, q3: str):
    """ Run research with the original query and the user's answers """
    answers_text = ""
    if q1 and ans1:
        answers_text += f"Q: {q1}\nA: {ans1}\n\n"
    if q2 and ans2:
        answers_text += f"Q: {q2}\nA: {ans2}\n\n"
    if q3 and ans3:
        answers_text += f"Q: {q3}\nA: {ans3}\n\n"
    report_md = await run_deep_research(query, answers_text.strip())
    return report_md


with gr.Blocks(theme=gr.themes.Default(primary_hue="sky")) as ui:
    gr.Markdown("# Deep Research")

    query_textbox = gr.Textbox(label="What topic would you like to research?")
    clarify_button = gr.Button("Get Clarifying Questions", variant="secondary")

    q1_state = gr.State("")
    q2_state = gr.State("")
    q3_state = gr.State("")

    with gr.Column(visible=False) as answers_section:
        gr.Markdown("### Please answer these questions to help focus the research:")
        ans1 = gr.Textbox(label="Q1", visible=False)
        ans2 = gr.Textbox(label="Q2", visible=False)
        ans3 = gr.Textbox(label="Q3", visible=False)

    run_button = gr.Button("Run Research", variant="primary", visible=False)
    report = gr.Markdown(label="Report")

    clarify_button.click(
        fn=clarify,
        inputs=[query_textbox],
        outputs=[ans1, ans2, ans3, q1_state, q2_state, q3_state, answers_section, run_button],
    )

    run_button.click(
        fn=run_research,
        inputs=[query_textbox, ans1, ans2, ans3, q1_state, q2_state, q3_state],
        outputs=[report],
    )

ui.launch(inbrowser=True)
